In [7]:
%pip install -qU langchain-openrouter langchain-community pydantic pypdf

In [8]:
import os
import getpass

api_key = getpass.getpass("Paste your NEW OpenRouter API key: ").strip()

os.environ["OPENROUTER_API_KEY"] = api_key

print("Key loaded:", bool(os.environ.get("OPENROUTER_API_KEY")))
print("Key starts correctly:", os.environ["OPENROUTER_API_KEY"].startswith("sk-or-v1-"))

Paste your NEW OpenRouter API key: ··········
Key loaded: True
Key starts correctly: True


In [9]:
from langchain_openrouter import ChatOpenRouter

MODEL_NAME = "nvidia/nemotron-3-nano-30b-a3b:free"

low_temp_model = ChatOpenRouter(
    model=MODEL_NAME,
    temperature=0.1,
    max_tokens=500,
    max_retries=2,
    openrouter_api_key=os.environ["OPENROUTER_API_KEY"],
)

high_temp_model = ChatOpenRouter(
    model=MODEL_NAME,
    temperature=1.2,
    max_tokens=500,
    max_retries=2,
    openrouter_api_key=os.environ["OPENROUTER_API_KEY"],
)

In [10]:
test_response = low_temp_model.invoke("Say hello in one short sentence.")
print(test_response.content)

Hello!


In [11]:
prompt = """
Answer this question as JSON only.

Question: What is Artificial Intelligence?

Style instruction:
Make it short and poetic.

Required JSON format:
{
  "answer": "...",
  "style": "..."
}
"""

In [12]:
low_response = low_temp_model.invoke(prompt)
high_response = high_temp_model.invoke(prompt)

print("LOW TEMPERATURE RESPONSE:")
print(low_response.content)

print("\nHIGH TEMPERATURE RESPONSE:")
print(high_response.content)

LOW TEMPERATURE RESPONSE:
{
  "answer": "A mindof code, dreaming in data.",
  "style": "short poetic"
}

HIGH TEMPERATURE RESPONSE:
{
  "answer": "A silent spark of code that dreams the world anew.",
  "style": "short poetic"
}


In [16]:
from typing import Literal, Dict, Any
from pydantic import BaseModel, Field
import json
import re

In [17]:
def extract_json(text):
    """
    Extract JSON object from model response.
    Works even if model returns ```json ... ```
    """
    text = text.strip()

    # remove markdown json fences
    text = text.replace("```json", "").replace("```", "").strip()

    # try direct json
    try:
        return json.loads(text)
    except:
        pass

    # extract first JSON object
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        return json.loads(match.group(0))

    raise ValueError("No valid JSON found in response:\n" + text)


def ask_json(model, prompt, pydantic_model):
    response = model.invoke(prompt)
    data = extract_json(response.content)
    return pydantic_model.model_validate(data)

In [18]:
class SentimentResult(BaseModel):
    sentiment: Literal["positive", "neutral", "negative"] = Field(
        description="The sentiment of the sentence"
    )

In [19]:
sentences = [
    "Kindness creates lasting joy.",
    "Success rewards persistent effort.",
    "I love Sunlight. It warms the skin.",
    "Pessemestic all the time.",
    "The storm caused damage!",
    "The clock ticks steadily.",
]

for sentence in sentences:
    prompt = f"""
    Classify the sentiment of the sentence.

    Sentence:
    {sentence}

    Return JSON only in this exact format:
    {{
      "sentiment": "positive"
    }}

    sentiment must be one of:
    positive, neutral, negative
    """

    result = ask_json(low_temp_model, prompt, SentimentResult)
    print(sentence, "=>", result.sentiment)

Kindness creates lasting joy. => positive
Success rewards persistent effort. => positive
I love Sunlight. It warms the skin. => positive
Pessemestic all the time. => negative
The storm caused damage! => negative
The clock ticks steadily. => neutral


In [20]:
Tag = Literal["cars", "shopping", "sports", "study", "work"]

class CategorizationResult(BaseModel):
    tags: list[Tag] = Field(
        description="One or more tags that best describe the sentence"
    )

In [21]:
texts = [
    "That restoration looks incredible; you have a real talent for mechanics.",
    "I found the perfect gift today! The staff was incredibly helpful.",
    "Great game today! Your teamwork was the key to that victory.",
    "Learning together helped me finally grasp these concepts. Thank you!",
]

for text in texts:
    prompt = f"""
    Categorize this sentence using only these tags:
    cars, shopping, sports, study, work

    Sentence:
    {text}

    Return JSON only in this exact format:
    {{
      "tags": ["cars"]
    }}

    The tags field must be a list.
    """

    result = ask_json(low_temp_model, prompt, CategorizationResult)
    print(text, "=>", result.tags)

That restoration looks incredible; you have a real talent for mechanics. => ['cars']
I found the perfect gift today! The staff was incredibly helpful. => ['shopping']
Great game today! Your teamwork was the key to that victory. => ['sports']
Learning together helped me finally grasp these concepts. Thank you! => ['study']


In [23]:
from langchain_community.document_loaders import PyPDFLoader
from pathlib import Path

CV_FILE = "cv.pdf"

if not Path(CV_FILE).exists():
    raise FileNotFoundError("Upload your CV file and name it cv.pdf")

loader = PyPDFLoader(CV_FILE)
docs = loader.load()

cv_text = "\n\n".join(doc.page_content for doc in docs)

print(cv_text[:1500])

Abdulrahman Yasser Solymani
Jeddah, Saudi Arabia abdulrahman.y.solymani@gmail.com +966 54 100 7757
Profile
Information Systems undergraduate at King Abdulaziz University with GPA 4.51/5.00, graduating in 2 months. Built
a full stack graduation project platform using React, Vite, and Supabase with AI assisted development. Completed
training in IT support within a hospital environment. Strong in system analysis, database design, and problem solving.
Education
King Abdulaziz UniversityJun 2022 – Aug 2026
Bachelor of Computer Information Systems
GPA:4.51 / 5.00
Experience
IT Support Trainee — King Abdulaziz University HospitalJun 2025 – Aug 2025
•Provided support for hardware, software, and system issues
•Assisted in user account setup and system configuration
•Gained practical exposure to IT support workflows in a hospital environment
Projects
Graduation Project Platform (GPP)2025 – Present
fcitgpp.site
•Developed a full stack platform for managing graduation projects and evaluations
•Imp

In [24]:
class ExperienceItem(BaseModel):
    title: str | None = Field(default=None)
    company: str | None = Field(default=None)
    dates: str | None = Field(default=None)
    description: str | None = Field(default=None)

class ResumeInfo(BaseModel):
    candidate_name: str | None = Field(default=None)
    email: str | None = Field(default=None)
    phone: str | None = Field(default=None)
    skills: list[str] = Field(default_factory=list)
    experience: list[ExperienceItem] = Field(default_factory=list)
    education: list[str] = Field(default_factory=list)

In [27]:
cv_model = ChatOpenRouter(
    model=MODEL_NAME,
    temperature=0.1,
    max_tokens=1500,
    max_retries=2,
    openrouter_api_key=os.environ["OPENROUTER_API_KEY"],
)

In [29]:
short_cv_text = cv_text[:5000]

resume_prompt = f"""
You are an information extraction assistant.

Extract information from this CV.

CV TEXT:
{short_cv_text}

Return JSON only. Do not explain anything.

Use exactly this JSON structure:

{{
  "candidate_name": null,
  "email": null,
  "phone": null,
  "skills": [],
  "experience": [],
  "education": []
}}

Rules:
- candidate_name can be string or null
- email can be string or null
- phone can be string or null
- skills must be a list of strings
- experience must be a list
- education must be a list of strings
"""

In [30]:
response = cv_model.invoke(resume_prompt)

print("RAW MODEL RESPONSE:")
print(response.content)

RAW MODEL RESPONSE:
{
  "candidate_name": "Abdulrahman Yasser Solymani",
  "email": "abdulrahman.y.solymani@gmail.com",
  "phone": "+966 54 100 7757",
  "skills": [
    "Java",
    "Python",
    "React",
    "Vite",
    "HTML",
    "CSS",
    "JavaScript",
    "Supabase",
    "REST APIs",
    "AI assisted development",
    "Excel",
    "Power BI",
    "Data Analysis",
    "Technical Support",
    "System Configuration",
    "Database Design",
    "System Analysis",
    "3D Modeling"
  ],
  "experience": [
    "IT Support Trainee at King Abdulaziz University Hospital (Jun 2025 – Aug 2025)"
  ],
  "education": [
    "King Abdulaziz University (Jun 2022 – Aug 2026) - Bachelor of Computer Information Systems, GPA: 4.51/5.00"
  ]
}


In [32]:
data = extract_json(response.content)

# تأكد أن المفاتيح الأساسية موجودة
data.setdefault("candidate_name", None)
data.setdefault("email", None)
data.setdefault("phone", None)
data.setdefault("skills", [])
data.setdefault("experience", [])
data.setdefault("education", [])

# إصلاح experience إذا رجعت كنصوص بدل dictionaries
fixed_experience = []

for item in data["experience"]:
    if isinstance(item, dict):
        fixed_experience.append({
            "title": item.get("title"),
            "company": item.get("company"),
            "dates": item.get("dates"),
            "description": item.get("description"),
        })
    elif isinstance(item, str):
        fixed_experience.append({
            "title": None,
            "company": None,
            "dates": None,
            "description": item,
        })

data["experience"] = fixed_experience

# إصلاح skills لو رجعت نص بدل list
if isinstance(data["skills"], str):
    data["skills"] = [data["skills"]]

# إصلاح education لو رجعت نص بدل list
if isinstance(data["education"], str):
    data["education"] = [data["education"]]

resume_result = ResumeInfo.model_validate(data)

resume_result

ResumeInfo(candidate_name='Abdulrahman Yasser Solymani', email='abdulrahman.y.solymani@gmail.com', phone='+966 54 100 7757', skills=['Java', 'Python', 'React', 'Vite', 'HTML', 'CSS', 'JavaScript', 'Supabase', 'REST APIs', 'AI assisted development', 'Excel', 'Power BI', 'Data Analysis', 'Technical Support', 'System Configuration', 'Database Design', 'System Analysis', '3D Modeling'], experience=[ExperienceItem(title=None, company=None, dates=None, description='IT Support Trainee at King Abdulaziz University Hospital (Jun 2025 – Aug 2025)')], education=['King Abdulaziz University (Jun 2022 – Aug 2026) - Bachelor of Computer Information Systems, GPA: 4.51/5.00'])

In [33]:
print("Candidate Name:", resume_result.candidate_name)
print("Email:", resume_result.email)
print("Phone:", resume_result.phone)
print("Skills:", resume_result.skills)
print("Experience:", resume_result.experience)
print("Education:", resume_result.education)

Candidate Name: Abdulrahman Yasser Solymani
Email: abdulrahman.y.solymani@gmail.com
Phone: +966 54 100 7757
Skills: ['Java', 'Python', 'React', 'Vite', 'HTML', 'CSS', 'JavaScript', 'Supabase', 'REST APIs', 'AI assisted development', 'Excel', 'Power BI', 'Data Analysis', 'Technical Support', 'System Configuration', 'Database Design', 'System Analysis', '3D Modeling']
Experience: [ExperienceItem(title=None, company=None, dates=None, description='IT Support Trainee at King Abdulaziz University Hospital (Jun 2025 – Aug 2025)')]
Education: ['King Abdulaziz University (Jun 2022 – Aug 2026) - Bachelor of Computer Information Systems, GPA: 4.51/5.00']


In [34]:
def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply(a, b):
    return a * b

def divide(a, b):
    if b == 0:
        raise ValueError("Cannot divide by zero.")
    return a / b

In [35]:
class ToolCall(BaseModel):
    tool_name: Literal["add", "subtract", "multiply", "divide"] = Field(
        description="The name of the function to use"
    )
    arguments: Dict[str, float] = Field(
        description="The parameters to pass to the function"
    )

In [36]:
system_prompt = """
You are a math assistant.

Available tools:

add(a, b): returns a + b
subtract(a, b): returns a - b
multiply(a, b): returns a * b
divide(a, b): returns a / b

Your job:
Choose the correct tool and return JSON only.

Return exactly this JSON format:
{
  "tool_name": "add",
  "arguments": {
    "a": 2,
    "b": 5
  }
}

Rules:
- tool_name must be one of: add, subtract, multiply, divide
- arguments must contain only two numbers: a and b
- Do not explain anything
- Return JSON only
"""

In [37]:
question = "what is two plus 5"

tool_prompt = f"""
{system_prompt}

User question:
{question}
"""

response = low_temp_model.invoke(tool_prompt)

print("RAW MODEL RESPONSE:")
print(response.content)

RAW MODEL RESPONSE:
{
  "tool_name": "add",
  "arguments": {
    "a": 2,
    "b": 5  }
}


In [38]:
data = extract_json(response.content)

# إصلاح بسيط لو الموديل رجّع arguments ناقصة أو كـ string
data.setdefault("tool_name", "add")
data.setdefault("arguments", {})

if isinstance(data["arguments"], str):
    data["arguments"] = extract_json(data["arguments"])

data["arguments"].setdefault("a", 0)
data["arguments"].setdefault("b", 0)

tool_call = ToolCall.model_validate(data)

tool_call

ToolCall(tool_name='add', arguments={'a': 2.0, 'b': 5.0})

In [39]:
tool_map = {
    "add": add,
    "subtract": subtract,
    "multiply": multiply,
    "divide": divide,
}

function_to_call = tool_map[tool_call.tool_name]

args = {
    "a": float(tool_call.arguments["a"]),
    "b": float(tool_call.arguments["b"]),
}

result = function_to_call(**args)

print("Question:", question)
print("Tool selected:", tool_call.tool_name)
print("Arguments:", args)
print("Result:", result)

Question: what is two plus 5
Tool selected: add
Arguments: {'a': 2.0, 'b': 5.0}
Result: 7.0


In [40]:
questions = [
    "what is 10 minus 3",
    "multiply 6 by 7",
    "divide 20 by 4",
    "add 100 and 250",
]

for q in questions:
    tool_prompt = f"""
    {system_prompt}

    User question:
    {q}
    """

    response = low_temp_model.invoke(tool_prompt)

    data = extract_json(response.content)

    data.setdefault("tool_name", "add")
    data.setdefault("arguments", {})

    if isinstance(data["arguments"], str):
        data["arguments"] = extract_json(data["arguments"])

    data["arguments"].setdefault("a", 0)
    data["arguments"].setdefault("b", 0)

    tool_call = ToolCall.model_validate(data)

    function_to_call = tool_map[tool_call.tool_name]

    args = {
        "a": float(tool_call.arguments["a"]),
        "b": float(tool_call.arguments["b"]),
    }

    result = function_to_call(**args)

    print("Question:", q)
    print("Tool:", tool_call.tool_name)
    print("Arguments:", args)
    print("Result:", result)
    print("-" * 40)

Question: what is 10 minus 3
Tool: subtract
Arguments: {'a': 10.0, 'b': 3.0}
Result: 7.0
----------------------------------------
Question: multiply 6 by 7
Tool: multiply
Arguments: {'a': 6.0, 'b': 7.0}
Result: 42.0
----------------------------------------
Question: divide 20 by 4
Tool: divide
Arguments: {'a': 20.0, 'b': 4.0}
Result: 5.0
----------------------------------------
Question: add 100 and 250
Tool: add
Arguments: {'a': 100.0, 'b': 250.0}
Result: 350.0
----------------------------------------
